# L1a: Values and Primitive Data Types

Every value in a computer program has a type. This lecture focuses on Julia's primitive values: how their types determine storage, interpretation, and valid operations.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Inspect a value's representation:__ Determine the type, storage size, and bit pattern of a Julia value using [`typeof(...)`](https://docs.julialang.org/en/v1/base/base/#Core.typeof), [`sizeof(...)`](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D), and [`bitstring(...)`](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring).
> * __Compare primitive types:__ Explain how integer, Boolean, and floating-point types use different widths and interpret their stored bits differently.
> * __Connect characters to representation:__ Distinguish a character, its Unicode code point, and the bytes used to store it.

Let's get started!
___


## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Everything in this notebook uses only Julia's `Base` library, so no external packages are required here. See [the Julia programming language documentation](https://docs.julialang.org/en/v1/) for the functions and types we use below.

___

## Primitive Data Types
Primitive data types are the basic building blocks a language provides. They are _atomic_: they are not composed of other types, and they hold simple values such as numbers, characters, and truth values.

> __Why does the type matter?__
>
> A __type__ is not a label the language attaches for bookkeeping. It fixes how many bytes a value occupies, how the bit pattern in those bytes is interpreted, and which operations the compiler or interpreter will permit. Two values with identical bits can denote entirely different numbers under two different types.

We start with [Integers](https://docs.julialang.org/en/v1/base/numbers/#Core.Int) and [the `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool), then turn to floating-point values and characters.

### Integer and Boolean Types
An __integer__ represents a whole number $x\in\mathbb{Z}$: positive, negative, or zero. Julia stores integers in a _fixed-width_ binary form, typically 32 or 64 bits, which is what the `32` and `64` in `Int32` and `Int64` refer to. A __boolean__ represents a truth value, either `true` or `false`.

We will ask three questions of each: what is its type, what bits are stored, and how much space does it take. Let's bind a whole number to `x::Int64` and start there.

In [ ]:
x = 2 |> Int64; # select a whole number ... -2, -1, 0, 1, 2, ...

Every Julia value carries its own type, and [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) reports it:

In [ ]:
typeof(x) # this returns the type of the argument

The type tells us how the bits are _interpreted_. [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) shows us the bits themselves.

> __Reading a bitstring:__
>
> The result has one character per bit, most significant bit first, so an `Int64` produces a 64-character string. This is the literal content of memory, not a decimal rendering of it. Only primitive types have a bitstring, because only they occupy a single contiguous block of fixed width.

So what is actually stored for `x`?

In [ ]:
bitstring(x) # shows the bit pattern stored in memory

Now the same three questions for a boolean. A variable of type `Bool` ranges over $\mathbb{B} = \left\{\text{true},\text{false}\right\}$, so it carries exactly one bit of information. Let's bind `false` to `flag::Bool`:

In [ ]:
flag = false; # the flag variable can take on values of {true | false}

The pattern is the same as before. First the type:

In [ ]:
typeof(flag)

Then the stored bits:

In [ ]:
bitstring(flag) # this should be 8 bits wide

A `Bool` carries a single bit of information, but memory is addressed in __bytes__. Let's use [the `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) to see how much space our `flag` variable actually takes up:

In [ ]:
sizeof(flag) # number of bytes used to store the Bool (not sizeof(x) -- x is an Int64!)

___

### Floating point types
Floating-point types model real numbers using three components according to [the IEEE 754 standard](https://en.wikipedia.org/wiki/IEEE_754): a sign bit, an exponent (the scale), and a significand. Only the _fractional part_ of the significand is stored in memory; its leading digit is implicit. We take a floating-point number apart bit by bit in `L1d`, where that implicit leading digit turns out to matter.

> __Julia versus Python floating point numbers__: Julia provides three standard IEEE-754 floating-point types that trade off precision for storage: `Float16` (half-precision), `Float32` (single-precision), and `Float64` (double-precision). Python's built-in `float` type is always 64-bit double-precision.

Let's look at a couple of examples. First, here's a 64-bit number (Julia's default):

In [ ]:
let
    x = 54.13; # default: in Julia, the default floating point number is 64-bit.
    bitstring(x)
end

The same numerical value stored in 32-bits has a different memory layout:

In [ ]:
let
    x = 54.13 |> Float32 # cast to Float32 (single precision), not Float64
    bitstring(x) # gives a string with the bit pattern
end

Fewer bits means less storage and, as we will see in `L1d`, less precision. `Float16` halves the width again:

In [ ]:
let
    x = 54.13 |> Float16 # cast to Float16 (half precision), not Float64
    sizeof(x) # returns number of bytes used to store x
end

___

### Character Types
Text on computers is composed of characters, and each character is associated with a unique integer called its __code point__. Traditional systems used [ASCII](https://en.wikipedia.org/wiki/ASCII) with one byte per character, while modern systems use [Unicode encodings like UTF-8 or UTF-16](https://en.wikipedia.org/wiki/Unicode) to represent a much wider range of characters.

> __What a `Char` actually is:__
>
> It is tempting to say characters "are" integers, but in Julia `Char <: Integer` is `false`. A [Char](https://docs.julialang.org/en/v1/base/strings/#Core.Char) is its own primitive type that _converts to and from_ integers.
>
> Character encodings define the mapping between textual symbols and numeric code points, enabling text to be stored and transmitted as bytes. Julia's `Char` is a 4-byte (32-bit) primitive, but the bits it stores are the character's __UTF-8 bytes__, left-aligned in the word, _not_ the code point. `UInt32(c)` converts to the code point; it does not simply reinterpret the bits. We will see the difference below.

Let's explore [the `Char` type in Julia](https://docs.julialang.org/en/v1/manual/unicode-input/) (notice the single quotes):

In [ ]:
c = '🍣' # example Unicode character in Julia. See: https://docs.julialang.org/en/v1/manual/unicode-input/

What is the code point (the unique integer) for the character `c`? We convert it with [the `UInt32(...)` constructor](https://docs.julialang.org/en/v1/base/numbers/#Core.UInt32):

In [ ]:
code = UInt32(c) # extract code point as UInt32 (4 x bytes)

__Stored bits versus code point.__ The callout above claimed a `Char` holds UTF-8 bytes rather than the code point. Let's check that claim directly:

In [ ]:
(codepoint = string(UInt32(c), base = 16, pad = 8), # what UInt32(c) converts to
 stored     = string(reinterpret(UInt32, c), base = 16, pad = 8)) # what is actually in the 4 bytes

_Hmmm, what?_ That's a strange-looking integer! The `code::UInt32` is a [hexadecimal number](https://en.wikipedia.org/wiki/Hexadecimal), i.e., a number written in base 16. The giveaway (which is a convention) is the `0x` prefix. We'll dig into these numbers and examine representations in different bases later.

Can we see the data that each byte contains? Yes! Let's use [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret) and break the 4 bytes into four 1-byte blocks!

In [ ]:
reinterpret(Tuple{UInt8, UInt8, UInt8, UInt8}, code) |> collect

This factors the 32-bit value into four 8-bit (1-byte) values. Notice we list bytes from least significant to most significant (right to left) on little‑endian hosts (this corresponds to how `bitstring` shows bits on such machines).  
> __Endianness:__ This ordering relates to [Endianness](https://en.wikipedia.org/wiki/Endianness), which describes how computers store the bytes of multi-byte values. In little-endian systems (like most x86/x86-64 and ARM machines), the least significant byte comes first in memory, while big-endian systems store the most significant byte first. So when we reinterpret 0x0001F363 as four UInt8s on a little-endian machine, we get: `[0x63,0xF3,0x01,0x00]`  

Any `isbits` value can be split into bytes this way, but that does __not__ make a `Char` a collection. [`isprimitivetype(Char)`](https://docs.julialang.org/en/v1/base/base/#Base.isprimitivetype) is `true`: it has a fixed width and no independently addressable elements. Collection types are genuinely different: they hold a variable number of elements you can index, add to, or remove. Let's look at those next.

___

## Looking ahead

Primitive values become useful when we organize them. In Lab `L1b`, we will choose among tuples, arrays, sets, and dictionaries, then build a custom composite type.

___


## Summary

Every Julia value carries a type that fixes how it is stored in memory and which operations are valid on it.

> __Key Takeaways:__
>
> * **Types determine interpretation:** A bit pattern has meaning only when paired with a type.
> * **Width is a design choice:** Integer and floating-point families trade storage width against range and precision.
> * **Text also has a representation:** A character's code point and its stored bytes are related but distinct views of the same value.

These primitive values are the building blocks for the collections and custom types used throughout the course.
___
